In [ ]:
%pip install matplotlib
%pip install scipy
%pip install statsmodels
%pip install python-docx


In [ ]:
import pandas as pd
import os

# 读取 .dta 格式的数据文件
file_path = 'csmar_0110sample.dta'

# 检查文件是否存在
if os.path.exists(file_path):
    print(f"正在读取文件: {file_path}\n")
    
    # 读取数据
    data = pd.read_stata(file_path)
    
    # 显示数据基本信息
    print("=" * 60)
    print("数据基本信息")
    print("=" * 60)
    print(f"数据形状: {data.shape[0]} 行 × {data.shape[1]} 列")
    print(f"\n列名 ({len(data.columns)} 个):")
    for i, col in enumerate(data.columns, 1):
        print(f"  {i}. {col}")
    
    print("\n" + "=" * 60)
    print("前5行数据预览:")
    print("=" * 60)
    print(data.head())
    
    print("\n" + "=" * 60)
    print("数据类型:")
    print("=" * 60)
    print(data.dtypes)
    
    print("\n" + "=" * 60)
    print("数据统计摘要:")
    print("=" * 60)
    print(data.describe())
    
else:
    print(f"错误: 找不到文件 {file_path}")


In [ ]:
# 详细分析数据结构
print("=" * 70)
print("数据结构详细分析")
print("=" * 70)

print(f"\n1. 数据规模:")
print(f"   - 总行数: {data.shape[0]:,}")
print(f"   - 总列数: {data.shape[1]}")
print(f"   - 数据大小: {data.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

print(f"\n2. 缺失值情况:")
missing = data.isnull().sum()
missing_pct = (missing / len(data) * 100).round(2)
missing_info = pd.DataFrame({
    '缺失数量': missing,
    '缺失比例(%)': missing_pct
})
missing_info = missing_info[missing_info['缺失数量'] > 0]
if len(missing_info) > 0:
    print(missing_info)
else:
    print("   ✓ 没有缺失值")

print(f"\n3. 唯一值统计:")
unique_counts = {}
for col in data.columns:
    unique_counts[col] = data[col].nunique()
unique_df = pd.DataFrame(list(unique_counts.items()), columns=['列名', '唯一值数量'])
print(unique_df.to_string(index=False))

print(f"\n4. 时间范围:")
if 'year' in data.columns:
    print(f"   - 年份范围: {int(data['year'].min())} - {int(data['year'].max())}")
if 'month' in data.columns:
    print(f"   - 月份范围: {int(data['month'].min())} - {int(data['month'].max())}")

print(f"\n5. 股票数量:")
if 'stkcd' in data.columns:
    print(f"   - 唯一股票代码数: {data['stkcd'].nunique():,}")

print(f"\n6. 各列含义推测（基于列名）:")
column_meanings = {
    'stkcd': '股票代码 (Stock Code)',
    'year': '年份',
    'month': '月份',
    'ret': '收益率 (Return)',
    'size': '公司规模 (市值)',
    'r11': '可能为滞后11期的收益率',
    'bm': '账面市值比 (Book-to-Market Ratio)',
    'ep': '市盈率倒数 (Earnings-to-Price Ratio)',
    'roe': '净资产收益率 (Return on Equity)',
    'ivff': '可能为波动率指标',
    'beta': 'Beta系数（市场风险）',
    'tur': '换手率 (Turnover)',
    'srev': '可能为销售增长率或收益相关指标'
}
for col in data.columns:
    meaning = column_meanings.get(col, '未知')
    dtype = str(data[col].dtype)
    print(f"   - {col:8s}: {meaning:30s} ({dtype})")


In [ ]:
# ========================================
# 缺失值处理
# ========================================

print("=" * 70)
print("缺失值处理")
print("=" * 70)

# 1. 查看原始数据量
print(f"\n原始数据: {len(data):,} 行")

# 2. 显示缺失值情况
print("\n缺失值情况:")
missing_summary = pd.DataFrame({
    '缺失数量': data.isnull().sum(),
    '缺失比例(%)': (data.isnull().sum() / len(data) * 100).round(2)
})
print(missing_summary[missing_summary['缺失数量'] > 0])

# ========================================
# 处理策略：针对 IVOL 研究
# ========================================
# 策略：分层处理
# - 关键变量 (ivff, ret, size, year, month, stkcd): 缺失则删除
# - 其他控制变量: 用横截面中位数填充

print("\n" + "=" * 70)
print("处理策略:")
print("=" * 70)

# 创建数据副本
data_clean = data.copy()

# 步骤1: 删除关键变量缺失的观测值
key_vars = ['stkcd', 'year', 'month', 'ret', 'ivff', 'size']
print(f"\n步骤1: 删除关键变量 {key_vars} 中有缺失值的观测")
before = len(data_clean)
data_clean = data_clean.dropna(subset=key_vars)
after = len(data_clean)
print(f"   删除了 {before - after:,} 行 ({(before-after)/before*100:.2f}%)")
print(f"   剩余 {after:,} 行")

# 步骤2: 用横截面中位数填充其他控制变量
print(f"\n步骤2: 用横截面中位数填充其他控制变量")

# 按年月分组，用每个横截面的中位数填充
control_vars = ['r11', 'bm', 'ep', 'roe', 'beta', 'tur', 'srev']
for var in control_vars:
    if data_clean[var].isnull().sum() > 0:
        print(f"   - 处理 {var}...")
        # 用每个月的中位数填充
        data_clean[var] = data_clean.groupby(['year', 'month'])[var].transform(
            lambda x: x.fillna(x.median())
        )
        # 如果还有缺失（某些月份整体缺失），用全局中位数填充
        if data_clean[var].isnull().sum() > 0:
            data_clean[var].fillna(data_clean[var].median(), inplace=True)

print(f"\n处理完成:")
print(f"   剩余数据: {len(data_clean):,} 行")
print(f"   剩余缺失值: {data_clean.isnull().sum().sum()} 个")

# 最终清洗数据
data_final = data_clean.copy()

print("\n" + "=" * 70)
print("最终数据集:")
print("=" * 70)
print(f"  - 原始数据: {len(data):,} 行")
print(f"  - 清洗后数据: {len(data_final):,} 行")
print(f"  - 保留比例: {len(data_final)/len(data)*100:.2f}%")
print(f"  - 缺失值数量: {data_final.isnull().sum().sum()}")

# 验证关键变量无缺失
print(f"\n关键变量检查:")
for var in key_vars:
    missing = data_final[var].isnull().sum()
    print(f"  - {var}: {missing} 个缺失值 {'✓' if missing == 0 else '✗'}")


In [ ]:
# ========================================
# IVOL 分布特征分析
# ========================================

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

# 配置中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'STHeiti', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False  # 正常显示负号

print("=" * 70)
print("异质性波动率（IVOL）分布特征分析")
print("=" * 70)

# 1. 描述性统计
print("\n1. IVOL 描述性统计:")
print("-" * 70)
ivol_stats = data_final['ivff'].describe(percentiles=[.01, .05, .10, .25, .50, .75, .90, .95, .99])
print(ivol_stats)

# 计算偏度和峰度
skewness = stats.skew(data_final['ivff'])
kurtosis = stats.kurtosis(data_final['ivff'])
print(f"\n偏度 (Skewness): {skewness:.4f}")
print(f"峰度 (Kurtosis): {kurtosis:.4f}")

# 2. 按年份统计
print("\n2. IVOL 按年份统计:")
print("-" * 70)
yearly_stats = data_final.groupby('year')['ivff'].agg([
    ('均值', 'mean'),
    ('中位数', 'median'),
    ('标准差', 'std'),
    ('最小值', 'min'),
    ('最大值', 'max')
]).round(4)
print(yearly_stats)

# 3. 可视化分析
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle('异质性波动率（IVOL）分布特征分析', fontsize=16, fontweight='bold')

# 3.1 直方图 + 核密度估计
ax1 = axes[0, 0]
ax1.hist(data_final['ivff'], bins=50, density=True, alpha=0.7, color='steelblue', edgecolor='black')
from scipy.stats import gaussian_kde
kde = gaussian_kde(data_final['ivff'])
x_range = np.linspace(data_final['ivff'].min(), data_final['ivff'].max(), 100)
ax1.plot(x_range, kde(x_range), 'r-', linewidth=2, label='核密度估计')
ax1.set_xlabel('IVOL', fontsize=11)
ax1.set_ylabel('密度', fontsize=11)
ax1.set_title('IVOL 分布直方图', fontsize=12, fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 3.2 箱线图（按年份）
ax2 = axes[0, 1]
yearly_data = [data_final[data_final['year'] == y]['ivff'].values for y in sorted(data_final['year'].unique())]
box = ax2.boxplot(yearly_data, labels=[int(y) for y in sorted(data_final['year'].unique())], patch_artist=True)
for patch in box['boxes']:
    patch.set_facecolor('lightblue')
ax2.set_xlabel('年份', fontsize=11)
ax2.set_ylabel('IVOL', fontsize=11)
ax2.set_title('IVOL 按年份分布（箱线图）', fontsize=12, fontweight='bold')
ax2.grid(True, alpha=0.3, axis='y')
ax2.tick_params(axis='x', rotation=45)

# 3.3 时间序列趋势（月度平均）
ax3 = axes[1, 0]
monthly_avg = data_final.groupby(['year', 'month'])['ivff'].mean().reset_index()
ax3.plot(range(len(monthly_avg)), monthly_avg['ivff'], linewidth=1.5, color='darkblue')
ax3.set_xlabel('时间', fontsize=11)
ax3.set_ylabel('IVOL 均值', fontsize=11)
ax3.set_title('IVOL 时间序列趋势', fontsize=12, fontweight='bold')
ax3.grid(True, alpha=0.3)

# 3.4 Q-Q图（正态性检验）
ax4 = axes[1, 1]
stats.probplot(data_final['ivff'], dist="norm", plot=ax4)
ax4.set_title('IVOL Q-Q 图（正态性检验）', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# 4. 与其他变量的相关性
print("\n3. IVOL 与其他变量的相关系数:")
print("-" * 70)
corr_vars = ['ret', 'size', 'r11', 'bm', 'ep', 'roe', 'beta', 'tur', 'srev']
correlations = data_final[['ivff'] + corr_vars].corr()['ivff'][1:].sort_values(ascending=False)
for var, corr_val in correlations.items():
    print(f"  {var:8s}: {corr_val:7.4f}")

# 5. 异常值分析
print("\n4. 异常值分析:")
print("-" * 70)
Q1 = data_final['ivff'].quantile(0.25)
Q3 = data_final['ivff'].quantile(0.75)
IQR = Q3 - Q1
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
outliers = data_final[(data_final['ivff'] < lower_bound) | (data_final['ivff'] > upper_bound)]
print(f"  IQR 方法检测异常值:")
print(f"  - 下界: {lower_bound:.6f}")
print(f"  - 上界: {upper_bound:.6f}")
print(f"  - 异常值数量: {len(outliers):,} ({len(outliers)/len(data_final)*100:.2f}%)")

print("\n" + "=" * 70)
print("分布特征分析完成")
print("=" * 70)


In [ ]:
# ========================================
# 投资组合排序策略
# ========================================

print("=" * 70)
print("基于 IVOL 的投资组合排序策略")
print("=" * 70)

# 策略说明
print("\n策略设计:")
print("-" * 70)
print("1. 每月根据 IVOL 将股票分为 5 组（五分位数组合）")
print("2. 计算每组的等权重和市值加权收益率")
print("3. 构建多空组合：Low IVOL (组1) - High IVOL (组5)")
print("4. 检验 IVOL 异象在中国市场的表现")

# 参数设置
n_portfolios = 5  # 分为5组

# 创建投资组合
print(f"\n开始构建 {n_portfolios} 个投资组合...")

# 按月对股票进行 IVOL 排序和分组
def assign_portfolio(group, n_portfolios=5):
    """
    将每个横截面（月份）的股票按 IVOL 分组
    """
    # 使用 qcut 进行等频分组（每组股票数量尽量相等）
    try:
        group['ivol_portfolio'] = pd.qcut(group['ivff'], q=n_portfolios, labels=range(1, n_portfolios+1), duplicates='drop')
    except:
        # 如果分组失败（例如有太多重复值），使用 cut
        group['ivol_portfolio'] = pd.cut(group['ivff'], bins=n_portfolios, labels=range(1, n_portfolios+1))
    return group

# 按年月分组，对每个月的股票进行排序分组
data_portfolios = data_final.groupby(['year', 'month']).apply(assign_portfolio, n_portfolios=n_portfolios).reset_index(drop=True)

# 检查分组结果
print("\n分组统计:")
print("-" * 70)
portfolio_counts = data_portfolios['ivol_portfolio'].value_counts().sort_index()
print("各组股票数量:")
for port, count in portfolio_counts.items():
    print(f"  组 {port}: {count:,} 个观测值")

# 计算每个组合的收益率
print("\n计算投资组合收益率...")

# 1. 等权重收益率
ew_returns = data_portfolios.groupby(['year', 'month', 'ivol_portfolio'])['ret'].mean().reset_index()
ew_returns.columns = ['year', 'month', 'portfolio', 'ret_ew']

# 2. 市值加权收益率
# 计算权重：每个股票在其组内的市值权重
def calc_vw_return(group):
    """计算市值加权收益率"""
    weights = group['size'] / group['size'].sum()
    vw_ret = (group['ret'] * weights).sum()
    return vw_ret

vw_returns = data_portfolios.groupby(['year', 'month', 'ivol_portfolio']).apply(calc_vw_return).reset_index()
vw_returns.columns = ['year', 'month', 'portfolio', 'ret_vw']

# 合并等权重和市值加权收益率
portfolio_returns = ew_returns.merge(vw_returns, on=['year', 'month', 'portfolio'])

print(f"  完成！共计算 {len(portfolio_returns)} 个组合-月份的收益率")

# 展示样本数据
print("\n投资组合收益率样本（前10行）:")
print("-" * 70)
print(portfolio_returns.head(10).to_string(index=False))

# 保存结果供后续分析使用
print("\n投资组合构建完成！")
print("=" * 70)


In [ ]:
# ========================================
# 计算投资组合表现指标
# ========================================

print("=" * 70)
print("投资组合表现分析")
print("=" * 70)

# 函数：计算表现指标
def calculate_performance_metrics(returns, annualization_factor=12):
    """
    计算投资组合的表现指标
    
    参数:
    - returns: 收益率序列
    - annualization_factor: 年化因子（月度数据用12）
    
    返回:
    - 包含各项指标的字典
    """
    mean_ret = returns.mean() * annualization_factor * 100  # 年化收益率(%)
    std_ret = returns.std() * np.sqrt(annualization_factor) * 100  # 年化标准差(%)
    t_stat = returns.mean() / (returns.std() / np.sqrt(len(returns)))  # t统计量
    sharpe = (returns.mean() / returns.std()) * np.sqrt(annualization_factor)  # 夏普比率
    
    # 最大回撤
    cumulative = (1 + returns).cumprod()
    running_max = cumulative.expanding().max()
    drawdown = (cumulative - running_max) / running_max
    max_dd = drawdown.min() * 100  # 最大回撤(%)
    
    return {
        '年化收益率(%)': mean_ret,
        '年化标准差(%)': std_ret,
        't统计量': t_stat,
        '夏普比率': sharpe,
        '最大回撤(%)': max_dd,
        '观测数': len(returns)
    }

# 1. 计算各组合的表现指标
print("\n1. 各投资组合表现指标:")
print("-" * 70)

# 等权重组合表现
print("\n【等权重组合】")
ew_performance = []
for port in range(1, n_portfolios + 1):
    port_returns = portfolio_returns[portfolio_returns['portfolio'] == port]['ret_ew']
    metrics = calculate_performance_metrics(port_returns)
    metrics['组合'] = f"P{port} (IVOL {port})"
    ew_performance.append(metrics)

ew_performance_df = pd.DataFrame(ew_performance)
ew_performance_df = ew_performance_df[['组合', '年化收益率(%)', '年化标准差(%)', 't统计量', '夏普比率', '最大回撤(%)', '观测数']]
print(ew_performance_df.to_string(index=False))

# 市值加权组合表现
print("\n【市值加权组合】")
vw_performance = []
for port in range(1, n_portfolios + 1):
    port_returns = portfolio_returns[portfolio_returns['portfolio'] == port]['ret_vw']
    metrics = calculate_performance_metrics(port_returns)
    metrics['组合'] = f"P{port} (IVOL {port})"
    vw_performance.append(metrics)

vw_performance_df = pd.DataFrame(vw_performance)
vw_performance_df = vw_performance_df[['组合', '年化收益率(%)', '年化标准差(%)', 't统计量', '夏普比率', '最大回撤(%)', '观测数']]
print(vw_performance_df.to_string(index=False))

# 2. 构建并分析多空组合 (Low IVOL - High IVOL)
print("\n" + "=" * 70)
print("2. 多空组合表现（Low IVOL - High IVOL）:")
print("-" * 70)

# 等权重多空组合
ew_long_short = portfolio_returns.groupby(['year', 'month']).apply(
    lambda x: x[x['portfolio'] == 1]['ret_ew'].values[0] - x[x['portfolio'] == n_portfolios]['ret_ew'].values[0]
).reset_index()
ew_long_short.columns = ['year', 'month', 'ret_ls']

# 市值加权多空组合
vw_long_short = portfolio_returns.groupby(['year', 'month']).apply(
    lambda x: x[x['portfolio'] == 1]['ret_vw'].values[0] - x[x['portfolio'] == n_portfolios]['ret_vw'].values[0]
).reset_index()
vw_long_short.columns = ['year', 'month', 'ret_ls']

print("\n【等权重多空组合】")
ew_ls_metrics = calculate_performance_metrics(ew_long_short['ret_ls'])
for key, value in ew_ls_metrics.items():
    print(f"  {key:20s}: {value:10.4f}")

print("\n【市值加权多空组合】")
vw_ls_metrics = calculate_performance_metrics(vw_long_short['ret_ls'])
for key, value in vw_ls_metrics.items():
    print(f"  {key:20s}: {value:10.4f}")

# 3. IVOL 异象检验
print("\n" + "=" * 70)
print("3. IVOL 异象检验:")
print("-" * 70)

# 检验收益率是否单调递减（从低IVOL到高IVOL）
print("\n平均月度收益率 (%):")
print(f"  等权重: P1={ew_performance[0]['年化收益率(%)']/12:.4f}, P5={ew_performance[4]['年化收益率(%)']/12:.4f}")
print(f"  市值加权: P1={vw_performance[0]['年化收益率(%)']/12:.4f}, P5={vw_performance[4]['年化收益率(%)']/12:.4f}")

# 判断异象方向
ew_anomaly = "负向" if ew_performance[0]['年化收益率(%)'] < ew_performance[4]['年化收益率(%)'] else "正向"
vw_anomaly = "负向" if vw_performance[0]['年化收益率(%)'] < vw_performance[4]['年化收益率(%)'] else "正向"

print(f"\n异象方向:")
print(f"  等权重: {ew_anomaly} (低IVOL收益 {'<' if ew_anomaly == '负向' else '>'} 高IVOL收益)")
print(f"  市值加权: {vw_anomaly} (低IVOL收益 {'<' if vw_anomaly == '负向' else '>'} 高IVOL收益)")

# 多空组合显著性
print(f"\n多空组合显著性检验:")
print(f"  等权重 t统计量: {ew_ls_metrics['t统计量']:.4f} {'***' if abs(ew_ls_metrics['t统计量']) > 2.576 else '**' if abs(ew_ls_metrics['t统计量']) > 1.96 else '*' if abs(ew_ls_metrics['t统计量']) > 1.645 else ''}")
print(f"  市值加权 t统计量: {vw_ls_metrics['t统计量']:.4f} {'***' if abs(vw_ls_metrics['t统计量']) > 2.576 else '**' if abs(vw_ls_metrics['t统计量']) > 1.96 else '*' if abs(vw_ls_metrics['t统计量']) > 1.645 else ''}")
print(f"\n  注: *** p<0.01, ** p<0.05, * p<0.10")

print("\n" + "=" * 70)
print("表现指标计算完成！")
print("=" * 70)


In [ ]:
# ========================================
# 投资组合表现可视化
# ========================================

print("=" * 70)
print("投资组合表现可视化")
print("=" * 70)

# 配置中文字体
plt.rcParams['font.sans-serif'] = ['Arial Unicode MS', 'SimHei', 'STHeiti', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

fig, axes = plt.subplots(2, 2, figsize=(15, 11))
fig.suptitle('基于 IVOL 的投资组合表现分析', fontsize=16, fontweight='bold')

# 1. 各组合年化收益率对比
ax1 = axes[0, 0]
x_pos = np.arange(n_portfolios)
ew_rets = [ew_performance[i]['年化收益率(%)'] for i in range(n_portfolios)]
vw_rets = [vw_performance[i]['年化收益率(%)'] for i in range(n_portfolios)]

width = 0.35
ax1.bar(x_pos - width/2, ew_rets, width, label='等权重', color='steelblue', alpha=0.8)
ax1.bar(x_pos + width/2, vw_rets, width, label='市值加权', color='coral', alpha=0.8)
ax1.set_xlabel('IVOL 组合', fontsize=11)
ax1.set_ylabel('年化收益率 (%)', fontsize=11)
ax1.set_title('各组合年化收益率对比', fontsize=12, fontweight='bold')
ax1.set_xticks(x_pos)
ax1.set_xticklabels([f'P{i+1}' for i in range(n_portfolios)])
ax1.legend()
ax1.grid(True, alpha=0.3, axis='y')
ax1.axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# 2. 夏普比率对比
ax2 = axes[0, 1]
ew_sharpe = [ew_performance[i]['夏普比率'] for i in range(n_portfolios)]
vw_sharpe = [vw_performance[i]['夏普比率'] for i in range(n_portfolios)]

ax2.bar(x_pos - width/2, ew_sharpe, width, label='等权重', color='steelblue', alpha=0.8)
ax2.bar(x_pos + width/2, vw_sharpe, width, label='市值加权', color='coral', alpha=0.8)
ax2.set_xlabel('IVOL 组合', fontsize=11)
ax2.set_ylabel('夏普比率', fontsize=11)
ax2.set_title('各组合夏普比率对比', fontsize=12, fontweight='bold')
ax2.set_xticks(x_pos)
ax2.set_xticklabels([f'P{i+1}' for i in range(n_portfolios)])
ax2.legend()
ax2.grid(True, alpha=0.3, axis='y')
ax2.axhline(y=0, color='black', linestyle='-', linewidth=0.8)

# 3. 多空组合累计收益率
ax3 = axes[1, 0]

# 计算累计收益
ew_cumret = (1 + ew_long_short['ret_ls']).cumprod()
vw_cumret = (1 + vw_long_short['ret_ls']).cumprod()

ax3.plot(range(len(ew_cumret)), (ew_cumret - 1) * 100, label='等权重', linewidth=2, color='steelblue')
ax3.plot(range(len(vw_cumret)), (vw_cumret - 1) * 100, label='市值加权', linewidth=2, color='coral')
ax3.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)

ax3.set_xlabel('月份', fontsize=11)
ax3.set_ylabel('累计收益率 (%)', fontsize=11)
ax3.set_title('多空组合累计收益率 (Low IVOL - High IVOL)', fontsize=12, fontweight='bold')
ax3.legend()
ax3.grid(True, alpha=0.3)

# 4. 风险-收益散点图
ax4 = axes[1, 1]
ew_stds = [ew_performance[i]['年化标准差(%)'] for i in range(n_portfolios)]
for i in range(n_portfolios):
    color = 'darkgreen' if i == 0 else 'darkred' if i == n_portfolios-1 else 'gray'
    ax4.scatter(ew_stds[i], ew_rets[i], s=150, alpha=0.7, color=color, edgecolors='black', linewidths=1.5)
    ax4.annotate(f'P{i+1}', (ew_stds[i], ew_rets[i]), fontsize=9, ha='center', va='center', fontweight='bold', color='white')

ax4.set_xlabel('年化标准差 (%)', fontsize=11)
ax4.set_ylabel('年化收益率 (%)', fontsize=11)
ax4.set_title('风险-收益散点图（等权重）', fontsize=12, fontweight='bold')
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n可视化完成！")
print("=" * 70)


In [ ]:
# ========================================
# 构建 Fama-French 三因子（类 CH-3）
# ========================================

print("=" * 70)
print("构建 Fama-French 三因子模型")
print("=" * 70)

# 使用清洗后的数据
factor_data = data_final.copy()

# 1. 构建市场因子 (MKT)
print("\n1. 构建市场因子 (MKT)...")
mkt_factor = factor_data.groupby(['year', 'month']).apply(
    lambda x: (x['ret'] * x['size']).sum() / x['size'].sum()  # 市值加权
).reset_index()
mkt_factor.columns = ['year', 'month', 'MKT']
print(f"   完成！市场因子均值: {mkt_factor['MKT'].mean()*100:.4f}%")

# 2. 构建规模因子 (SMB: Small Minus Big)
print("\n2. 构建规模因子 (SMB)...")

def calc_smb(group):
    """计算 SMB 因子"""
    # 按市值分为大小两组
    median_size = group['size'].median()
    small = group[group['size'] <= median_size]
    big = group[group['size'] > median_size]
    
    # 等权收益
    small_ret = small['ret'].mean()
    big_ret = big['ret'].mean()
    
    return small_ret - big_ret

smb_factor = factor_data.groupby(['year', 'month']).apply(calc_smb).reset_index()
smb_factor.columns = ['year', 'month', 'SMB']
print(f"   完成！SMB 因子均值: {smb_factor['SMB'].mean()*100:.4f}%")

# 3. 构建价值因子 (HML: High Minus Low book-to-market)
print("\n3. 构建价值因子 (HML)...")

def calc_hml(group):
    """计算 HML 因子"""
    # 去除缺失值
    valid = group.dropna(subset=['bm'])
    if len(valid) < 10:  # 样本过少则返回 NaN
        return np.nan
    
    # 按 BM 分为高低两组（30%/70%分位）
    low_bm = valid['bm'].quantile(0.3)
    high_bm = valid['bm'].quantile(0.7)
    
    low = valid[valid['bm'] <= low_bm]
    high = valid[valid['bm'] >= high_bm]
    
    # 等权收益
    low_ret = low['ret'].mean() if len(low) > 0 else np.nan
    high_ret = high['ret'].mean() if len(high) > 0 else np.nan
    
    return high_ret - low_ret

hml_factor = factor_data.groupby(['year', 'month']).apply(calc_hml).reset_index()
hml_factor.columns = ['year', 'month', 'HML']
print(f"   完成！HML 因子均值: {hml_factor['HML'].mean()*100:.4f}%")

# 4. 合并所有因子
factors = mkt_factor.merge(smb_factor, on=['year', 'month'])
factors = factors.merge(hml_factor, on=['year', 'month'])

print("\n" + "=" * 70)
print("因子统计摘要:")
print("=" * 70)
print("\n因子年化收益率和波动率:")
factor_stats = pd.DataFrame({
    '年化收益率(%)': factors[['MKT', 'SMB', 'HML']].mean() * 12 * 100,
    '年化波动率(%)': factors[['MKT', 'SMB', 'HML']].std() * np.sqrt(12) * 100,
    '夏普比率': (factors[['MKT', 'SMB', 'HML']].mean() / factors[['MKT', 'SMB', 'HML']].std()) * np.sqrt(12)
})
print(factor_stats.round(4))

# 4. 构建盈利因子 (PMU: Profitable Minus Unprofitable)
print("\n4. 构建盈利因子 (PMU)...")

def calc_pmu(group):
    """计算盈利因子（基于 ROE）"""
    # 去除缺失值
    valid = group.dropna(subset=['roe'])
    if len(valid) < 10:  # 样本过少则返回 NaN
        return np.nan
    
    # 按 ROE 分为高低两组（30%/70%分位）
    low_roe = valid['roe'].quantile(0.3)
    high_roe = valid['roe'].quantile(0.7)
    
    low = valid[valid['roe'] <= low_roe]
    high = valid[valid['roe'] >= high_roe]
    
    # 等权收益
    low_ret = low['ret'].mean() if len(low) > 0 else np.nan
    high_ret = high['ret'].mean() if len(high) > 0 else np.nan
    
    return high_ret - low_ret

pmu_factor = factor_data.groupby(['year', 'month']).apply(calc_pmu).reset_index()
pmu_factor.columns = ['year', 'month', 'PMU']
print(f"   完成！PMU 因子均值: {pmu_factor['PMU'].mean()*100:.4f}%")

# 5. 合并所有因子（CH-4）
factors = factors.merge(pmu_factor, on=['year', 'month'])

print("\n" + "=" * 70)
print("因子统计摘要:")
print("=" * 70)
print("\n因子年化收益率和波动率:")
factor_stats = pd.DataFrame({
    '年化收益率(%)': factors[['MKT', 'SMB', 'HML', 'PMU']].mean() * 12 * 100,
    '年化波动率(%)': factors[['MKT', 'SMB', 'HML', 'PMU']].std() * np.sqrt(12) * 100,
    '夏普比率': (factors[['MKT', 'SMB', 'HML', 'PMU']].mean() / factors[['MKT', 'SMB', 'HML', 'PMU']].std()) * np.sqrt(12)
})
print(factor_stats.round(4))

# 6. 因子相关性
print("\n因子相关性矩阵:")
corr_matrix = factors[['MKT', 'SMB', 'HML', 'PMU']].corr()
print(corr_matrix.round(4))

print("\n" + "=" * 70)
print("四因子构建完成！(CH-4模型)")
print("=" * 70)


In [ ]:
# ========================================
# 因子模型回归分析
# ========================================

import statsmodels.api as sm
from scipy import stats as scipy_stats

print("=" * 70)
print("因子模型回归分析：检验 IVOL 异象")
print("=" * 70)

# 合并多空组合收益率和因子数据
regression_data = ew_long_short.merge(factors, on=['year', 'month'])

# 去除缺失值
regression_data = regression_data.dropna()

print(f"\n回归样本: {len(regression_data)} 个月")

# ===========================================
# 1. CAPM 模型回归
# ===========================================
print("\n" + "=" * 70)
print("模型 1: CAPM (单因子模型)")
print("=" * 70)
print("回归方程: R_portfolio = α + β_MKT × MKT + ε\n")

X_capm = sm.add_constant(regression_data['MKT'])
y = regression_data['ret_ls']

model_capm = sm.OLS(y, X_capm).fit()

print(model_capm.summary())

# 提取关键结果
alpha_capm = model_capm.params['const'] * 12 * 100  # 年化 alpha (%)
alpha_t_capm = model_capm.tvalues['const']
r2_capm = model_capm.rsquared

print(f"\n关键结果:")
print(f"  Alpha (年化):  {alpha_capm:7.4f}% (t = {alpha_t_capm:6.3f})")
print(f"  R-squared:     {r2_capm:7.4f}")

# ===========================================
# 2. Fama-French 三因子模型回归
# ===========================================
print("\n" + "=" * 70)
print("模型 2: Fama-French 三因子模型")
print("=" * 70)
print("回归方程: R_portfolio = α + β_MKT × MKT + β_SMB × SMB + β_HML × HML + ε\n")

X_ff3 = sm.add_constant(regression_data[['MKT', 'SMB', 'HML']])
y = regression_data['ret_ls']

model_ff3 = sm.OLS(y, X_ff3).fit()

print(model_ff3.summary())

# 提取关键结果
alpha_ff3 = model_ff3.params['const'] * 12 * 100  # 年化 alpha (%)
alpha_t_ff3 = model_ff3.tvalues['const']
r2_ff3 = model_ff3.rsquared

print(f"\n关键结果:")
print(f"  Alpha (年化):  {alpha_ff3:7.4f}% (t = {alpha_t_ff3:6.3f})")
print(f"  R-squared:     {r2_ff3:7.4f}")

# ===========================================
# 3. CH-4 四因子模型回归
# ===========================================
print("\n" + "=" * 70)
print("模型 3: CH-4 四因子模型")
print("=" * 70)
print("回归方程: R_portfolio = α + β_MKT × MKT + β_SMB × SMB + β_HML × HML + β_PMU × PMU + ε\n")

X_ch4 = sm.add_constant(regression_data[['MKT', 'SMB', 'HML', 'PMU']])
y = regression_data['ret_ls']

model_ch4 = sm.OLS(y, X_ch4).fit()

print(model_ch4.summary())

# 提取关键结果
alpha_ch4 = model_ch4.params['const'] * 12 * 100  # 年化 alpha (%)
alpha_t_ch4 = model_ch4.tvalues['const']
r2_ch4 = model_ch4.rsquared

print(f"\n关键结果:")
print(f"  Alpha (年化):  {alpha_ch4:7.4f}% (t = {alpha_t_ch4:6.3f})")
print(f"  R-squared:     {r2_ch4:7.4f}")

# ===========================================
# 4. 结果对比表格
# ===========================================
print("\n" + "=" * 70)
print("回归结果汇总")
print("=" * 70)

# 构建结果表格
results_summary = pd.DataFrame({
    'CAPM': [
        f"{model_capm.params['const']*100:.4f}",
        f"({model_capm.tvalues['const']:.3f})",
        f"{model_capm.params['MKT']:.4f}",
        f"({model_capm.tvalues['MKT']:.3f})",
        '-',
        '-',
        '-',
        '-',
        '-',
        '-',
        f"{model_capm.rsquared:.4f}"
    ],
    'FF-3': [
        f"{model_ff3.params['const']*100:.4f}",
        f"({model_ff3.tvalues['const']:.3f})",
        f"{model_ff3.params['MKT']:.4f}",
        f"({model_ff3.tvalues['MKT']:.3f})",
        f"{model_ff3.params['SMB']:.4f}",
        f"({model_ff3.tvalues['SMB']:.3f})",
        f"{model_ff3.params['HML']:.4f}",
        f"({model_ff3.tvalues['HML']:.3f})",
        '-',
        '-',
        f"{model_ff3.rsquared:.4f}"
    ],
    'CH-4': [
        f"{model_ch4.params['const']*100:.4f}",
        f"({model_ch4.tvalues['const']:.3f})",
        f"{model_ch4.params['MKT']:.4f}",
        f"({model_ch4.tvalues['MKT']:.3f})",
        f"{model_ch4.params['SMB']:.4f}",
        f"({model_ch4.tvalues['SMB']:.3f})",
        f"{model_ch4.params['HML']:.4f}",
        f"({model_ch4.tvalues['HML']:.3f})",
        f"{model_ch4.params['PMU']:.4f}",
        f"({model_ch4.tvalues['PMU']:.3f})",
        f"{model_ch4.rsquared:.4f}"
    ]
}, index=['Alpha (%)', '(t-stat)', 'MKT', '(t-stat)', 'SMB', '(t-stat)', 'HML', '(t-stat)', 'PMU', '(t-stat)', 'R²'])

print("\n多空组合 (Low IVOL - High IVOL) 回归结果:")
print(results_summary.to_string())

# ===========================================
# 4. 解读
# ===========================================
print("\n" + "=" * 70)
print("结果解读:")
print("=" * 70)

# Alpha 显著性判断
def significance_stars(t_stat):
    abs_t = abs(t_stat)
    if abs_t > 2.576:
        return "***"
    elif abs_t > 1.96:
        return "**"
    elif abs_t > 1.645:
        return "*"
    else:
        return ""

print(f"\n1. CAPM 模型:")
print(f"   - Alpha = {alpha_capm:.4f}% {significance_stars(alpha_t_capm)}")
if alpha_t_capm > 1.96:
    print(f"   - IVOL 异象在控制市场风险后**仍然显著**")
else:
    print(f"   - IVOL 异象在控制市场风险后不显著")

print(f"\n2. FF-3 模型:")
print(f"   - Alpha = {alpha_ff3:.4f}% {significance_stars(alpha_t_ff3)}")
if alpha_t_ff3 > 1.96:
    print(f"   - IVOL 异象在控制市场、规模、价值因子后**仍然显著**")
    print(f"   - 说明 IVOL 异象不能被经典因子完全解释")
else:
    print(f"   - IVOL 异象可以被经典因子解释")

print(f"\n3. CH-4 模型:")
print(f"   - Alpha = {alpha_ch4:.4f}% {significance_stars(alpha_t_ch4)}")
if alpha_t_ch4 > 1.96:
    print(f"   - IVOL 异象在控制市场、规模、价值、盈利因子后**仍然显著**")
    print(f"   - 说明 IVOL 异象是独立的，具有增量解释力")
else:
    print(f"   - IVOL 异象可以被四因子模型解释")

print(f"\n4. 模型比较:")
print(f"   - CAPM R² = {r2_capm:.4f}")
print(f"   - FF-3 R² = {r2_ff3:.4f}")
print(f"   - CH-4 R² = {r2_ch4:.4f}")
if r2_ch4 > r2_ff3:
    print(f"   - CH-4 模型的解释力最强，盈利因子有增量作用")
elif r2_ff3 > r2_capm:
    print(f"   - FF-3 模型的解释力更强")

print("\n注: *** p<0.01, ** p<0.05, * p<0.10")
print("=" * 70)


In [ ]:
# ========================================
# 结果解读和经济含义
# ========================================

print("=" * 70)
print("因子模型回归结果的经济含义")
print("=" * 70)

print("""
一、Alpha（截距项）的含义
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Alpha 是控制风险因子后的超额收益，代表策略的"真实"盈利能力。

【正向 Alpha】(α > 0)
  ✓ 低 IVOL 股票的收益 > 高 IVOL 股票（正向异象）
  ✓ 买入低波动、卖出高波动的策略能赚钱
  ✓ 与美国市场的发现一致（Ang et al. 2006）

【负向 Alpha】(α < 0)  
  ✓ 高 IVOL 股票的收益 > 低 IVOL 股票（负向异象）
  ✓ 买入高波动、卖出低波动的策略能赚钱
  ✓ 与理论预期一致（高风险高回报）

【Alpha 显著性】
  - |t| > 2.576 (***): 1% 水平显著，结果非常可靠
  - |t| > 1.96  (**):  5% 水平显著，结果较为可靠
  - |t| > 1.645 (*):  10% 水平显著，结果弱显著
  - |t| < 1.645:      不显著，可能是随机波动


二、CAPM vs FF-3 模型对比
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

【情况 1】CAPM Alpha 显著，FF-3 Alpha 也显著
  → IVOL 异象是真实存在的
  → 不能被市场、规模、价值因子解释
  → 可能反映了市场定价错误或其他风险
  → **投资价值：高**（策略有超额收益）

【情况 2】CAPM Alpha 显著，FF-3 Alpha 不显著
  → IVOL 异象可以被规模或价值因子解释
  → 可能是小市值效应或价值溢价的体现
  → **投资价值：中等**（策略收益可用其他因子替代）

【情况 3】两个模型 Alpha 都不显著
  → IVOL 异象不存在，或样本期内不显著
  → 低/高 IVOL 股票收益差异可以用因子解释
  → **投资价值：低**（无超额收益）


三、因子系数（β）的含义
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

【β_MKT (市场因子)】
  - β_MKT > 0: 多空组合与市场同向波动
  - β_MKT < 0: 多空组合与市场反向波动
  - |β_MKT| 越大，市场风险暴露越高

【β_SMB (规模因子)】
  - β_SMB > 0: 策略偏向小市值股票
  - β_SMB < 0: 策略偏向大市值股票
  - 显著说明低/高 IVOL 股票的市值有系统性差异

【β_HML (价值因子)】
  - β_HML > 0: 策略偏向价值股（高 BM）
  - β_HML < 0: 策略偏向成长股（低 BM）
  - 显著说明低/高 IVOL 股票的估值特征不同


四、R² (拟合优度) 的含义
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

R² 衡量因子模型能解释多少策略收益的变动

【R² 高 (>0.5)】
  → 因子模型解释力强
  → 策略收益主要来自已知风险因子
  → Alpha 更可能是真实的超额收益

【R² 低 (<0.3)】
  → 因子模型解释力弱
  → 策略收益可能来自其他未知因素
  → 或者策略收益波动较大、不稳定


五、实际投资含义
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

【如果 Alpha 显著为正】
  ✓ 低 IVOL 策略有投资价值
  ✓ 可以构建实际投资组合
  ✓ 但需要考虑交易成本和市场冲击

【如果 Alpha 显著为负】  
  ✓ 高 IVOL 策略有投资价值
  ✓ 可能反映投资者非理性行为
  ✓ 或高 IVOL 股票有彩票偏好溢价

【如果 Alpha 不显著】
  ✗ IVOL 策略无超额收益
  ✗ 不建议单独使用 IVOL 作为选股因子
  ✗ 可以结合其他因子使用


六、与文献对比
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

【美国市场 (Ang et al. 2006, 2009)】
  - IVOL 异象：负向（高 IVOL 低收益）
  - Alpha 显著为正（Low - High）
  - 被称为"IVOL 之谜"

【中国市场（你的研究）】
  - 如果结果与美国一致 → 异象具有普遍性
  - 如果结果相反 → 市场特征差异
  - 如果结果不显著 → 可能是样本期或市场环境影响

""")

print("=" * 70)
print("建议：结合上述解读框架分析你的回归结果")
print("=" * 70)
